# 04 — Zero-Shot Contextomy Detection

Benchmarks four zero-shot approaches on the 100-article English hand-annotated test set.

**Task:** Given a headline quote + best-matching body sentence (+ context), predict `1 = contextomized`, `0 = modified`.

**Target:** Song et al. (2023) QuoteCSE — AUC=0.768 (Korean, supervised).

**Approaches:**
1. Cosine threshold — `best_sim` from Notebook 02, fixed τ=0.5
2. NLI (mDeBERTa) — contradiction score as contextomization probability (Burnham, 2025)
3. GPT-4o direct — classification prompt (Zheng et al., 2023)
4. GPT-4o CoT — chain-of-thought prompt (Wei et al., 2022)

**Primary metric:** AUC. F1 reported at τ=0.5 for all approaches.

## 0. Setup

In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import f1_score, roc_auc_score, classification_report
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

THRESHOLD = 0.5  # fixed decision threshold for all approaches

df = pd.read_csv('data/test_set_100.csv')
df = df.dropna(subset=['label']).copy()
df['label'] = df['label'].astype(int)
for col in ['prev_sentence', 'next_sentence']:
    df[col] = df[col].fillna('')

y_true = df['label'].values

print(f"Test set: {len(df)} articles")
print(f"Contextomized: {y_true.sum()}  |  Modified: {(y_true==0).sum()}")
print(f"Sim bins: {df['sim_bin'].value_counts().to_dict()}")

Test set: 100 articles
Contextomized: 15  |  Modified: 85
Sim bins: {'mid': 36, 'low': 33, 'high': 31}


In [2]:
def build_premise(row):
    """Concatenate context window: prev + best body sentence + next."""
    parts = [row['prev_sentence'], row['best_body_sentence'], row['next_sentence']]
    return ' '.join(p for p in parts if p).strip()

def evaluate(y_true, scores, threshold=THRESHOLD):
    """Compute AUC and F1 at fixed threshold. Primary metric is AUC."""
    preds = (scores >= threshold).astype(int)
    auc = roc_auc_score(y_true, scores)
    f1  = f1_score(y_true, preds, zero_division=0)
    print(f"AUC : {auc:.3f}  |  F1 (τ={threshold}) : {f1:.3f}")
    print(classification_report(y_true, preds, target_names=['modified', 'contextomized']))
    return preds, auc, f1

---
## Approach 1 — Cosine Similarity Threshold (τ=0.5)

The cosine similarity between the headline quote embedding and the best-matching body sentence (`best_sim`, Notebook 02) is a direct signal: low similarity → likely contextomized. The score is inverted so that high values indicate contextomization. A fixed threshold τ=0.5 is applied consistently across all approaches.

In [3]:
cosine_scores = 1 - df['best_sim'].values  # invert: high = likely contextomized

cosine_preds, cosine_auc, cosine_f1 = evaluate(y_true, cosine_scores)

AUC : 0.629  |  F1 (τ=0.5) : 0.267
               precision    recall  f1-score   support

     modified       0.87      0.87      0.87        85
contextomized       0.27      0.27      0.27        15

     accuracy                           0.78       100
    macro avg       0.57      0.57      0.57       100
 weighted avg       0.78      0.78      0.78       100



---
## Approach 2 — NLI-based Detection (mDeBERTa)

Contextomy is a semantic faithfulness problem: a modified quote entails the body meaning; a contextomized quote contradicts or diverges from it. We operationalize this as NLI: body context = premise, headline quote = hypothesis. The multilingual `mDeBERTa-v3-base-mnli-xnli` model is applied zero-shot. The **contradiction probability** serves as the contextomization score.

Inference runs in batches on GPU if available, otherwise CPU.

In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import DataLoader

DEVICE    = 'cuda' if torch.cuda.is_available() else 'cpu'
NLI_MODEL = 'MoritzLaurer/mDeBERTa-v3-base-mnli-xnli'
BATCH_SIZE = 16  # reduce to 8 if you hit memory errors on GPU

print(f"Device: {DEVICE}")

tokenizer = AutoTokenizer.from_pretrained(NLI_MODEL)
nli_model = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL).to(DEVICE)
nli_model.eval()

# Confirm label order before running
print(nli_model.config.id2label)
# Expected: {0: 'contradiction', 1: 'neutral', 2: 'entailment'}
# Update CONTRADICTION_IDX in next cell if different

In [ ]:
CONTRADICTION_IDX = 0  # update if id2label above differs

premises    = [build_premise(row) for _, row in df.iterrows()]
hypotheses  = df['headline_quote'].tolist()

nli_scores = []
for i in range(0, len(premises), BATCH_SIZE):
    batch_p = premises[i:i+BATCH_SIZE]
    batch_h = hypotheses[i:i+BATCH_SIZE]
    inputs  = tokenizer(batch_p, batch_h, return_tensors='pt',
                        truncation=True, max_length=512, padding=True).to(DEVICE)
    with torch.no_grad():
        probs = F.softmax(nli_model(**inputs).logits, dim=-1).cpu().numpy()
    nli_scores.extend(probs[:, CONTRADICTION_IDX].tolist())
    print(f"  Batch {i//BATCH_SIZE + 1}/{-(-len(premises)//BATCH_SIZE)} done")

nli_scores = np.array(nli_scores)
nli_preds, nli_auc, nli_f1 = evaluate(y_true, nli_scores)

---
## Approach 3 — GPT-4o Zero-Shot (Direct Prompt)

Following the LLM-as-judge paradigm (Zheng et al., 2023), GPT-4o receives the headline quote and body context window and outputs a binary label and a probability estimate (0.0–1.0) used for AUC.

In [ ]:
import openai
import json
import os
import time

client = openai.OpenAI(api_key=os.environ['OPENAI_API_KEY'])

SYSTEM_DIRECT = """You are a journalism ethics expert specialising in quote accuracy.

Definitions:
- MODIFIED: The headline quote faithfully preserves the speaker's meaning. Minor editorial changes (grammar, brevity, pronoun replacement) are acceptable.
- CONTEXTOMIZED: The headline quote distorts the speaker's intended meaning by removing context, altering emphasis, or combining fragments in a misleading way.

You will receive a headline quote and the body text where it originated.
Output ONLY valid JSON with exactly two fields:
- "label": "CONTEXTOMIZED" or "MODIFIED"
- "probability": float 0.0-1.0, your estimated probability that the quote is contextomized"""

def classify_direct(row):
    user_msg = f'HEADLINE QUOTE: "{row["headline_quote"]}"\nBODY CONTEXT: "{build_premise(row)}"'
    response = client.chat.completions.create(
        model='gpt-4o',
        messages=[
            {'role': 'system', 'content': SYSTEM_DIRECT},
            {'role': 'user',   'content': user_msg}
        ],
        temperature=0,
        max_tokens=60
    )
    text   = response.choices[0].message.content.strip().replace('```json','').replace('```','')
    parsed = json.loads(text)
    return parsed['label'], float(parsed['probability'])

# Pilot on 5 rows before committing to full run
print("Pilot (5 rows):")
for _, row in df.head(5).iterrows():
    label, prob = classify_direct(row)
    print(f"  gt={row['label']}  pred={label}  prob={prob:.2f}  |  {row['headline_quote'][:70]}")
    time.sleep(0.3)

In [ ]:
gpt_labels, gpt_scores = [], []

for i, (_, row) in enumerate(df.iterrows()):
    try:
        label, prob = classify_direct(row)
        gpt_labels.append(1 if label == 'CONTEXTOMIZED' else 0)
        gpt_scores.append(prob)
    except Exception as e:
        print(f"Row {i} failed: {e}")
        gpt_labels.append(0)
        gpt_scores.append(0.5)
    if (i + 1) % 10 == 0:
        print(f"  {i+1}/100 done")
    time.sleep(0.3)

gpt_labels = np.array(gpt_labels)
gpt_scores = np.array(gpt_scores)
gpt_preds, gpt_auc, gpt_f1 = evaluate(y_true, gpt_scores)

---
## Approach 4 — GPT-4o Chain-of-Thought

Chain-of-thought prompting (Wei et al., 2022) elicits step-by-step reasoning before the final label. The model first states the speaker's intended meaning from the body context, then assesses whether the headline quote preserves it. This decomposition is suited to contextomy detection, where the distortion can be subtle.

In [ ]:
SYSTEM_COT = """You are a journalism ethics expert specialising in quote accuracy.

Definitions:
- MODIFIED: The headline quote faithfully preserves the speaker's meaning.
- CONTEXTOMIZED: The headline quote distorts the speaker's intended meaning by removing context, altering emphasis, or combining fragments in a misleading way.

Reason step by step, then output ONLY valid JSON with exactly four fields:
- "step1": What is the speaker's intended meaning in the body context? (1 sentence)
- "step2": Does the headline quote preserve or distort that meaning? (1 sentence)
- "label": "CONTEXTOMIZED" or "MODIFIED"
- "probability": float 0.0-1.0, your estimated probability that the quote is contextomized"""

def classify_cot(row):
    user_msg = f'HEADLINE QUOTE: "{row["headline_quote"]}"\nBODY CONTEXT: "{build_premise(row)}"'
    response = client.chat.completions.create(
        model='gpt-4o',
        messages=[
            {'role': 'system', 'content': SYSTEM_COT},
            {'role': 'user',   'content': user_msg}
        ],
        temperature=0,
        max_tokens=200
    )
    text   = response.choices[0].message.content.strip().replace('```json','').replace('```','')
    parsed = json.loads(text)
    return parsed['label'], float(parsed['probability'])

# Pilot on 5 rows
print("Pilot (5 rows):")
for _, row in df.head(5).iterrows():
    label, prob = classify_cot(row)
    print(f"  gt={row['label']}  pred={label}  prob={prob:.2f}  |  {row['headline_quote'][:70]}")
    time.sleep(0.3)

In [ ]:
cot_labels, cot_scores = [], []

for i, (_, row) in enumerate(df.iterrows()):
    try:
        label, prob = classify_cot(row)
        cot_labels.append(1 if label == 'CONTEXTOMIZED' else 0)
        cot_scores.append(prob)
    except Exception as e:
        print(f"Row {i} failed: {e}")
        cot_labels.append(0)
        cot_scores.append(0.5)
    if (i + 1) % 10 == 0:
        print(f"  {i+1}/100 done")
    time.sleep(0.3)

cot_labels = np.array(cot_labels)
cot_scores = np.array(cot_scores)
cot_preds, cot_auc, cot_f1 = evaluate(y_true, cot_scores)

---
## Results Summary

Comparison table mirroring Song et al. (2023) Table 2. AUC is the primary metric. F1 is reported at fixed τ=0.5.

In [ ]:
results = pd.DataFrame([
    {'Method': 'BERT (Song et al., Korean)',           'AUC': 0.662, 'F1': 0.665, 'Setting': 'supervised'},
    {'Method': 'BERT fine-tune (Song et al., Korean)', 'AUC': 0.749, 'F1': 0.754, 'Setting': 'supervised'},
    {'Method': 'QuoteCSE (Song et al., Korean)',       'AUC': 0.768, 'F1': 0.770, 'Setting': 'supervised'},
    {'Method': 'Cosine τ=0.5 (ours)',                  'AUC': round(cosine_auc, 3), 'F1': round(cosine_f1, 3), 'Setting': 'zero-shot'},
    {'Method': 'mDeBERTa NLI (ours)',                  'AUC': round(nli_auc, 3),    'F1': round(nli_f1, 3),    'Setting': 'zero-shot'},
    {'Method': 'GPT-4o direct (ours)',                 'AUC': round(gpt_auc, 3),    'F1': round(gpt_f1, 3),    'Setting': 'zero-shot'},
    {'Method': 'GPT-4o CoT (ours)',                    'AUC': round(cot_auc, 3),    'F1': round(cot_f1, 3),    'Setting': 'zero-shot'},
])

print(results.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
x, w = np.arange(len(results)), 0.35

ax.bar(x - w/2, results['AUC'], w, label='AUC', color='#DD8452', alpha=0.85)
ax.bar(x + w/2, results['F1'],  w, label='F1',  color='#4C72B0', alpha=0.85)

ax.axhline(0.768, color='#DD8452', linestyle='--', linewidth=0.9, label='QuoteCSE AUC (Song et al.)')
ax.axhline(0.770, color='#4C72B0', linestyle='--', linewidth=0.9, label='QuoteCSE F1 (Song et al.)')
ax.axvline(2.5,   color='grey',    linestyle=':',  linewidth=1)
ax.text(1.0, 0.42, 'Song et al. (Korean, supervised)', ha='center', fontsize=8, color='grey')
ax.text(4.5, 0.42, 'Ours (English, zero-shot)',         ha='center', fontsize=8, color='grey')

ax.set_xticks(x)
ax.set_xticklabels(results['Method'], rotation=25, ha='right', fontsize=8)
ax.set_ylim(0.4, 1.0)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))
ax.set_ylabel('Score')
ax.set_title('Zero-Shot Contextomy Detection — English vs. Song et al. (2023)')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig('data/results_comparison.png', dpi=150)
plt.show()

---
## Error Analysis

Error breakdown by similarity bin (low / mid / high) for the best-performing model. Reveals whether the model struggles more with subtle (mid-sim) or obvious (low-sim) contextomization.

In [ ]:
# Set to whichever approach scored highest AUC
best_preds  = cot_labels
best_scores = cot_scores

df_eval = df.copy()
df_eval['pred']    = best_preds
df_eval['correct'] = (df_eval['pred'] == df_eval['label']).astype(int)
df_eval['error_type'] = 'correct'
df_eval.loc[(df_eval['pred']==1) & (df_eval['label']==0), 'error_type'] = 'FP'
df_eval.loc[(df_eval['pred']==0) & (df_eval['label']==1), 'error_type'] = 'FN'

print("Accuracy by similarity bin:")
print(df_eval.groupby('sim_bin')['correct'].mean().round(3))
print()

fp = df_eval[df_eval['error_type'] == 'FP']
fn = df_eval[df_eval['error_type'] == 'FN']
print(f"False Positives (predicted contextomized, was modified) : {len(fp)}")
print(f"False Negatives (predicted modified, was contextomized) : {len(fn)}")

In [ ]:
cols = ['headline_quote', 'best_body_sentence', 'best_sim', 'sim_bin', 'label', 'notes']
fn[cols].to_csv('data/false_negatives.csv', index=False)
fn[cols]

---
## Save All Predictions

In [ ]:
df_eval['pred_cosine']  = cosine_preds
df_eval['pred_nli']     = nli_preds
df_eval['pred_gpt']     = gpt_preds
df_eval['pred_cot']     = cot_preds
df_eval['score_cosine'] = cosine_scores
df_eval['score_nli']    = nli_scores
df_eval['score_gpt']    = gpt_scores
df_eval['score_cot']    = cot_scores

df_eval.to_csv('data/test_set_predictions.csv', index=False)
print("Saved: data/test_set_predictions.csv")
print()
print(results.to_string(index=False))